# Design of experiments (DOE): sweeping the RLFT **reward shape**

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package. See
[`docs/notes/doe-and-visualization.md`](../docs/notes/doe-and-visualization.md).

The other DOE notebooks sweep tuning **hyperparameters**
([`12_doe_dpo_rlft.ipynb`](12_doe_dpo_rlft.ipynb) crosses `epochs` x
`samples_per_prompt`). This one sweeps the axis unique to RLFT — the **reward
function** — tuning `gemini-3.5-flash` on the same verifiable-math dataset once
per reward shape and scoring every result on the same held-out split against an
**untuned baseline**, so the before→after lift is explicit:

- `string-match` — declarative `Answer:\s*-?\d+` format reward (no sandbox)
- `code-exec` — ships `geap_tuning.rlft.reward` to the sandbox (correctness)
- `autorater` — an LLM judge grading explanation quality
- `composite` — code-exec (0.8) + autorater (0.2), a weighted blend

A reward config is a **non-scalar** object, so it cannot be a grid axis (grid
values feed the run slug and Experiments params). Following the established
pattern, each reward rides in `sweep.fixed` (filtered from the slug / rows /
Experiments params by `doe._scalar_params`). So each shape is its own **single-run
`SweepConfig`** (empty grid → one run), all logged to one shared Experiment; we
combine them under labels **the driver controls**, because every empty-grid
`RunSpec.name` is `"default"` and would otherwise collide.

> **Requires live GCP and incurs tuning cost** (this launches ~4 RLFT jobs). Have
> a real `.env` and `gcloud auth` in place; keep the tuning/Experiments region
> aligned. `gemini-3.5-flash` — verify region availability first. Charts need the
> optional viz group (`uv sync --group viz`).

In [ ]:
from geap_tuning.config import genai_client, load_config

cfg = load_config()
client = genai_client(cfg)  # tuning is regional-only; global excludes tuning
cfg

## 1. Build the dataset and the four reward shapes

The math RLFT dataset is staged to GCS (train/val); the test split is held out
locally. We then build one reward object per shape. The **autorater** judge needs
a fully-qualified publisher path (bare names fail with an opaque error), reused by
the `composite` blend.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.rlft.data import (
    MATH_PROBLEMS,
    build_rlft_dataset,
    build_rlft_records,
    split_dataset,
)
from geap_tuning.rlft.tune import (
    build_autorater_reward_config,
    build_composite_reward_config,
    build_reward_config,
    build_string_match_reward_config,
)

paths = build_rlft_dataset("../datasets/rlft_math")
train_uri = upload_file(paths["train"], f"{cfg.bucket}/doe_rlft_rewards/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/doe_rlft_rewards/val.jsonl")

train_problems, _, test_problems = split_dataset(MATH_PROBLEMS)
train_records = build_rlft_records(train_problems)
test_records = build_rlft_records(test_problems)

autorater_model = (
    f"projects/{cfg.project}/locations/{cfg.location}/publishers/google/models/gemini-2.5-flash"
)

# label -> (reward kwargs for launch_rlft_job / validate_reward_config)
reward_shapes = {
    "string-match": {"reward_config": build_string_match_reward_config()},
    "code-exec": {"reward_config": build_reward_config()},
    "autorater": {"reward_config": build_autorater_reward_config(autorater_model=autorater_model)},
    "composite": {
        "composite_reward_config": build_composite_reward_config(
            [
                (build_reward_config(), 0.8),
                (build_autorater_reward_config(autorater_model=autorater_model), 0.2),
            ]
        )
    },
}
list(reward_shapes)

## 2. Preflight every reward on one record

`validate_reward` scores each reward on a single example before we spend money —
RLFT auto-stops if >80% of reward calls fail, so a broken reward is worth catching
here. A non-null `error`/`NaN` means the reward is broken.

In [ ]:
from geap_tuning.rlft.tune import validate_reward_config

for label, kw in reward_shapes.items():
    preflight = validate_reward_config(
        client,
        project=cfg.project,
        location=cfg.location,
        sample_answer="Answer: 4",
        example_record=train_records[0],
        reward_config=kw.get("reward_config"),
        composite_reward_config=kw.get("composite_reward_config"),
    )
    print(f"[{label}] {preflight}")

## 3. Run each reward shape as its own single-run sweep

`method="RLFT"` selects `launch_rlft_job`; each shape is a `SweepConfig` with an
**empty grid** (one run) and its reward in `fixed`. All share one Experiment and
the same offline scorer (held-out answer accuracy, reward > 0 ⇒ correct). Reruns
reuse finished jobs via the deterministic display name `geap-doe-<name>-default`.

In [ ]:
from geap_tuning.doe import SweepConfig, run_sweep
from geap_tuning.experiments import init_experiment
from geap_tuning.inference import generate
from geap_tuning.rlft.evaluate import run_rlft_eval

EXPERIMENT_NAME = "geap-doe-rlft-rewards"
BASE_MODEL = "gemini-3.5-flash"
METRIC = "accuracy"

init_experiment(EXPERIMENT_NAME, project=cfg.project, location=cfg.location)


def evaluate_fn(endpoint: str) -> dict:
    return run_rlft_eval(
        test_records,
        generate_fn=lambda user_text, e=endpoint: generate(client, e, user_text),
    )


results = {}
for label, kw in reward_shapes.items():
    sweep = SweepConfig(name=f"rew-{label}", method="RLFT", base_model=BASE_MODEL, fixed=kw)
    result = run_sweep(
        client,
        sweep,
        train_uri=train_uri,
        val_uri=val_uri,
        evaluate_fn=evaluate_fn,
        experiment=EXPERIMENT_NAME,
        labels=cfg.labels,
    )[0]
    results[label] = result
    tag = "reused" if result.reused else "launched"
    print(f"  {label}: accuracy={result.metrics[METRIC]:.3f} ({tag})")

## 4. Untuned baseline + cross-reward comparison

The baseline scores the **untuned** `gemini-3.5-flash` on the same test split.
Gemini 3.x *inference* runs on the `global` endpoint, so we build a separate
global-routed client (`genai_client(cfg, base_model=...)`) — distinct from the
regional tuning client. We hand-build the comparison rows under our own labels
(bypassing `aggregate_results`, whose keys would all be `"default"`), then chart
with `plot_metric_bars`.

In [ ]:
from geap_tuning.viz import plot_metric_bars

base_client = genai_client(cfg, base_model=BASE_MODEL)
baseline = run_rlft_eval(
    test_records,
    generate_fn=lambda user_text: generate(base_client, BASE_MODEL, user_text),
)
print(f"untuned baseline: accuracy={baseline[METRIC]:.3f}")

rows = [{"run": "untuned", METRIC: baseline[METRIC]}]
rows += [{"run": label, METRIC: r.metrics[METRIC]} for label, r in results.items()]

best_label = max(results, key=lambda label: results[label].metrics[METRIC])
print(f"best reward shape ({METRIC}): {best_label} = {results[best_label].metrics[METRIC]:.3f}")
plot_metric_bars(rows, metric=METRIC)

## 5. Read the tracked runs back from Experiments

`experiment_dataframe` returns a pandas table matching **Agent Platform Studio →
Experiments** — the four tuned shapes (the untuned baseline is offline-only, not
an Experiments run). [`11_multi_run_viz.ipynb`](11_multi_run_viz.ipynb) charts
this experiment directly with **zero tuning cost** (`--experiment
geap-doe-rlft-rewards`).

In [ ]:
from geap_tuning.experiments import experiment_dataframe

experiment_dataframe(EXPERIMENT_NAME)